<a href="https://colab.research.google.com/github/Salma-Jedidi/python-projects/blob/yoga-project/Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import os
import numpy as np
import torch
import glob
import torch.nn as nn
from torchvision.transforms import transforms
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.autograd import Variable
import torchvision
import pathlib

In [ ]:
#checking fot device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
print(device)

cpu


In [ ]:
#Transforms
transformer=transforms.Compose(
    [
        transforms.Resize((150,150)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(), #changes the color from 0-255 to 0-1
        transforms.Normalize([0.5,0.5,0.5], #normalizee from 0-1 to -1-1, formula to create new pixules (x-mean)/std
                             [0.5,0.5,0.5])
    ]
)

In [ ]:
#Drive
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


In [ ]:
#Path
train_path = '/content/gdrive/MyDrive/DATASET/TRAIN'
test_path = '/content/gdrive/MyDrive/DATASET/TEST'

In [ ]:
#DataLoader
train_loader=DataLoader(
    torchvision.datasets.ImageFolder(train_path,transform=transformer),
    batch_size=50, shuffle=True
)
test_loader=DataLoader(
    torchvision.datasets.ImageFolder(test_path,transform=transformer),
    batch_size=50, shuffle=True
)

In [ ]:
#catagories
root=pathlib.Path(train_path)
classes = sorted([j.name.split('/')[-1] for j in root.iterdir()])

In [ ]:
print(classes)

['downdog', 'goddess', 'plank', 'tree', 'warrior2']


In [ ]:
#CNN Network


class ConvNet(nn.Module):
    def __init__(self,num_classes=5):
        super(ConvNet,self).__init__()

        #Output size after convolution filter
        #((w-f+2P)/s) +1

        #Input shape= (256,3,150,150)

        self.conv1=nn.Conv2d(in_channels=3,out_channels=12,kernel_size=3,stride=1,padding=1)
        #Shape= (50,12,150,150)
        self.bn1=nn.BatchNorm2d(num_features=12)
        #Shape= (50,12,150,150)
        self.relu1=nn.ReLU()
        #Shape= (50,12,150,150)

        self.pool=nn.MaxPool2d(kernel_size=2)
        #Reduce the image size be factor 2
        #Shape= (50,12,75,75)


        self.conv2=nn.Conv2d(in_channels=12,out_channels=20,kernel_size=3,stride=1,padding=1)
        #Shape= (50,20,75,75)
        self.relu2=nn.ReLU()
        #Shape= (50,20,75,75)



        self.conv3=nn.Conv2d(in_channels=20,out_channels=32,kernel_size=3,stride=1,padding=1)
        #Shape= (50,32,75,75)
        self.bn3=nn.BatchNorm2d(num_features=32)
        #Shape= (50,32,75,75)
        self.relu3=nn.ReLU()
        #Shape= (50,32,75,75)


        self.fc=nn.Linear(in_features=75 * 75 * 32,out_features=num_classes)



        #Feed forwad function

    def forward(self,input):
        output=self.conv1(input)
        output=self.bn1(output)
        output=self.relu1(output)

        output=self.pool(output)

        output=self.conv2(output)
        output=self.relu2(output)

        output=self.conv3(output)
        output=self.bn3(output)
        output=self.relu3(output)


            #Above output will be in matrix form, with shape (256,32,75,75)

        output=output.view(-1,32*75*75)


        output=self.fc(output)

        return output

In [ ]:
model = ConvNet(num_classes=5).to(device)

In [ ]:
#Optimizer and loss function
optimizer=Adam(model.parameters(),lr=0.001,weight_decay=0.0001)
loss_function=nn.CrossEntropyLoss()

In [ ]:
num_epochs=10

In [ ]:
#calculating the size of training and testing images
train_count=len(glob.glob(train_path+'/**/*.jpg'))
test_count=len(glob.glob(test_path+'/**/*.jpg'))

In [ ]:
print(train_count,test_count)

1010 427


In [ ]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [ ]:

#Model training and saving best model

best_accuracy=0.0

for epoch in range(num_epochs):

    #Evaluation and training on training dataset
    model.train()
    train_accuracy=0.0
    train_loss=0.0

    for i, (images,labels) in enumerate(train_loader):
        if torch.cuda.is_available():
            images=Variable(images.cuda())
            labels=Variable(labels.cuda())

        optimizer.zero_grad()

        outputs=model(images)
        loss=loss_function(outputs,labels)
        loss.backward()
        optimizer.step()


        train_loss+= loss.cpu().data*images.size(0)
        _,prediction=torch.max(outputs.data,1)

        train_accuracy+=int(torch.sum(prediction==labels.data))

    train_accuracy=train_accuracy/train_count
    train_loss=train_loss/train_count


    # Evaluation on testing dataset
    model.eval()

    test_accuracy=0.0
    for i, (images,labels) in enumerate(test_loader):

        outputs=model(images)
        _,prediction=torch.max(outputs.data,1)
        test_accuracy+=int(torch.sum(prediction==labels.data))

    test_accuracy=test_accuracy/test_count


    print('Epoch: '+str(epoch)+' Train Loss: '+str(train_loss)+' Train Accuracy: '+str(train_accuracy)+' Test Accuracy: '+str(test_accuracy))

    #Save the best model
    if test_accuracy>best_accuracy:
        torch.save(model.state_dict(),'best_checkpoint.model')
        best_accuracy=test_accuracy


Epoch: 0 Train Loss: tensor(25.7173) Train Accuracy: 0.3475247524752475 Test Accuracy: 0.43559718969555034
Epoch: 1 Train Loss: tensor(5.1571) Train Accuracy: 0.6475247524752475 Test Accuracy: 0.6861826697892272
Epoch: 2 Train Loss: tensor(1.6227) Train Accuracy: 0.8455445544554455 Test Accuracy: 0.7377049180327869
Epoch: 3 Train Loss: tensor(1.5170) Train Accuracy: 0.8732673267326733 Test Accuracy: 0.7611241217798594
Epoch: 4 Train Loss: tensor(0.7486) Train Accuracy: 0.9603960396039604 Test Accuracy: 0.7868852459016393
Epoch: 5 Train Loss: tensor(0.3974) Train Accuracy: 1.004950495049505 Test Accuracy: 0.8079625292740047
Epoch: 6 Train Loss: tensor(0.2785) Train Accuracy: 1.0297029702970297 Test Accuracy: 0.7447306791569087
Epoch: 7 Train Loss: tensor(0.2934) Train Accuracy: 1.0386138613861386 Test Accuracy: 0.7681498829039812
Epoch: 8 Train Loss: tensor(0.1894) Train Accuracy: 1.0326732673267327 Test Accuracy: 0.7423887587822015
Epoch: 9 Train Loss: tensor(0.1067) Train Accuracy: 1.